# Lab 23.3: The Scored Comparison

Chapter 23, Choosing, Measuring and Using XAI Tools (the chapter's case study). Run the cells in order. In section 4 you may change the weights and rerun; nothing else needs to be edited. Run Lab 23.2 first: this notebook reads its `test_drive_results.json`.

A bank's model risk team must approve a toolset for a gradient-boosted credit model. The credit team needs three forms of explanation: principal reasons for each decline, a global view for model documentation, and counterfactual suggestions that an applicant could act on. Four toolsets survived the filter:

- **A:** SHAP (TreeExplainer) for reasons and global views, DiCE for counterfactuals, on the approved gradient-boosted model;
- **B:** an Explainable Boosting Machine (InterpretML) replacing the model, with DiCE for counterfactuals;
- **C:** LIME for reasons, DiCE for counterfactuals;
- **D:** HermAI 0.1.8 for reasons, a surrogate tree and counterfactuals (disclosure: maintained by the book's author; scored by the same rules as the others).

The data case is invented; the measurements are real and reproducible.

In [1]:
import json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from interpret.glassbox import ExplainableBoostingClassifier
import dice_ml
from importlib.metadata import version
for p in ["scikit-learn", "dice-ml", "interpret-core", "hermai"]:
    print(p, version(p))
R = json.load(open("../lab2/test_drive_results.json"))

scikit-learn 1.7.2
dice-ml 0.12
interpret-core 0.7.8
hermai 0.1.8


## 1. Rebuild the model and the 20 declines (as in Lab 23.2)

In [2]:
d = fetch_openml("credit-g", version=1, as_frame=True).frame
y = (d.pop("class") == "bad").astype(int)
cats = [c for c in d.columns if str(d[c].dtype) == "category"]
X = d.copy()
for c in cats:
    X[c] = X[c].cat.codes
X = X.astype(float); cols = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0).fit(Xtr, ytr)
A = Xte.iloc[np.argsort(-gbm.predict_proba(Xte)[:, 1])[:20]]
ebm = ExplainableBoostingClassifier(random_state=0, feature_types=["nominal" if c in cats else "continuous" for c in cols]).fit(Xtr, ytr)
IMMUTABLE = ["age", "personal_status", "foreign_worker"]   # the credit team's list: not actionable or not allowed

## 2. Counterfactuals: valid, actionable, sparse?

For each decline we ask for counterfactuals that bring predicted risk below 0.5. We record whether at least one is valid, the share that change an immutable feature, and how many features they change.

In [3]:
dtr = Xtr.astype(int).copy(); dtr["bad"] = ytr.values
num = [c for c in cols if c not in cats]
D = dice_ml.Data(dataframe=dtr, continuous_features=num, outcome_name="bad")
class Wrap:
    def __init__(s, m): s.m = m
    def predict_proba(s, Z): return s.m.predict_proba(pd.DataFrame(Z, columns=cols).astype(float))
    def predict(s, Z): return s.m.predict(pd.DataFrame(Z, columns=cols).astype(float))
cf = {}
for name, model, vary in [("DiCE, all features", gbm, "all"),
                          ("DiCE, immutables fixed", gbm, [c for c in cols if c not in IMMUTABLE]),
                          ("DiCE on EBM, immutables fixed", ebm, [c for c in cols if c not in IMMUTABLE])]:
    exp = dice_ml.Dice(D, dice_ml.Model(model=Wrap(model), backend="sklearn"), method="random")
    valid, viol, nchg = [], [], []; t = time.time()
    for i in range(len(A)):
        q = A.iloc[[i]].astype(int)
        df = exp.generate_counterfactuals(q, total_CFs=3, desired_class=0, random_seed=i,
                                          features_to_vary=vary, verbose=False).cf_examples_list[0].final_cfs_df
        if df is None or len(df) == 0:
            valid.append(0); continue
        Z = df[cols].astype(float); valid.append(int((model.predict_proba(Z)[:, 1] < 0.5).any()))
        chg = Z.values != q.values.astype(float)
        viol.append(chg[:, [cols.index(c) for c in IMMUTABLE]].any(1).mean()); nchg.append(chg.sum(1).mean())
    cf[name] = dict(valid=np.mean(valid), viol=np.mean(viol), nchg=np.mean(nchg), t=(time.time() - t) / len(A))
from hermai.perturbations.tabular import TabularPerturbationGenerator
from hermai.explainers import LocalExplainer
gen = TabularPerturbationGenerator(categorical_features=cats); gen.fit(Xtr)
herm = LocalExplainer(gbm, gen); np.random.seed(0)
valid, viol, nchg = [], [], []; t = time.time()
for i in range(len(A)):
    e = herm.explain(A.iloc[i])
    if e.counterfactual is None:
        valid.append(0); continue
    z = e.counterfactual[cols].values.astype(float)
    valid.append(int(gbm.predict_proba(pd.DataFrame([z], columns=cols))[0, 1] < 0.5))
    chg = z != A.iloc[i].values
    viol.append(float(chg[[cols.index(c) for c in IMMUTABLE]].any())); nchg.append(chg.sum())
cf["HermAI counterfactual"] = dict(valid=np.mean(valid), viol=np.mean(viol), nchg=np.mean(nchg), t=(time.time() - t) / len(A))
pd.DataFrame(cf).T.round(3)

,valid,viol,nchg,t
"DiCE, all features",1.0,0.55,1.667,0.060
"DiCE, immutables fixed",1.0,0.00,1.800,0.064
"DiCE on EBM, immutables fixed",1.0,0.00,1.783,0.064
HermAI counterfactual,1.0,1.00,11.200,0.017


**Checkpoint 1.** Every method finds a valid counterfactual for all 20 declines. With every feature allowed to change, about half of DiCE's counterfactuals (0.55) touch age, personal status or foreign-worker status; with those features fixed, none do, and each counterfactual changes about 1.8 features. HermAI 0.1.8 has no option to fix features: its counterfactual is the closest flipped perturbation, and because every numeric feature is perturbed it changes an immutable feature every time and about 11 of 20 features in all.

## 3. Turn measurements into scores (0 to 5) by written rules

In [4]:
ag = dict(zip(R["agree_names"], R["agree"][0]))            # overlap with exact TreeSHAP reasons
st = {k: np.mean(v) for k, v in R["stability"].items()}
rt = R["runtime"]
cfs = lambda d: 5 * d["valid"] * (1 - d["viol"])
runtime_score = lambda s: 5 if s < 0.1 else 4 if s < 0.5 else 3 if s < 1 else 2 if s < 5 else 1
months_score = lambda m: 5 if m <= 6 else 4 if m <= 12 else 3 if m <= 24 else 2 if m <= 48 else 1
# months from the last release on PyPI to 2026-11-24 (checked that day): shap 1, dice-ml 16, interpret 8, lime 77, hermai 14
last = {"shap": 1, "dice-ml": 16, "interpret": 8, "lime": 77, "hermai": 14}
scores = pd.DataFrame({
    "Agreement with exact reasons": [5, 5, 5 * ag["LIME default"], 5 * ag["HermAI default"]],
    "Stability over seeds": [5, 5, 5 * st["LIME default"], 5 * st["HermAI default"]],
    "Actionable counterfactuals": [cfs(cf["DiCE, immutables fixed"]), cfs(cf["DiCE on EBM, immutables fixed"]),
                                   cfs(cf["DiCE, immutables fixed"]), cfs(cf["HermAI counterfactual"])],
    "Runtime for nightly batch": [runtime_score(rt["TreeSHAP"] + cf["DiCE, immutables fixed"]["t"]),
                                  runtime_score(rt["EBM local"] + cf["DiCE on EBM, immutables fixed"]["t"]),
                                  runtime_score(rt["LIME default"] + cf["DiCE, immutables fixed"]["t"]),
                                  runtime_score(rt["HermAI default"])],
    "Forms covered": [5 * 3 / 4, 5 * 4 / 4, 5 * 2 / 4, 5 * 3 / 4],   # of: reasons, global view, counterfactual, glass box
    "Works with the approved model": [5, 1, 5, 5],                   # B means a new model and a full revalidation
    "License for production": [5, 5, 5, 5],                          # MIT or BSD everywhere on this shortlist
    "Maintenance": [min(months_score(last["shap"]), months_score(last["dice-ml"])),
                    min(months_score(last["interpret"]), months_score(last["dice-ml"])),
                    min(months_score(last["lime"]), months_score(last["dice-ml"])), months_score(last["hermai"])],
    "Reproducibility": [5, 5, 4, 3],   # 5 deterministic core; 4 seed argument; 3 global random state only
}, index=["A: SHAP + DiCE", "B: EBM + DiCE", "C: LIME + DiCE", "D: HermAI"]).T
scores.round(2)

,A: SHAP + DiCE,B: EBM + DiCE,C: LIME + DiCE,D: HermAI
Agreement with exact reasons,5.00,5.0,2.29,1.38
Stability over seeds,5.00,5.0,4.27,4.29
Actionable counterfactuals,5.00,5.0,5.00,0.00
Runtime for nightly batch,5.00,5.0,5.00,5.00
Forms covered,3.75,5.0,2.50,3.75
Works with the approved model,5.00,1.0,5.00,5.00
License for production,5.00,5.0,5.00,5.00
Maintenance,3.00,3.0,1.00,3.00
Reproducibility,5.00,5.0,4.00,3.00


## 4. Weights, totals and a stress test

The weights are the model risk team's judgment. Change them here if you disagree, then rerun this cell and the next.

In [5]:
WEIGHTS = {"Agreement with exact reasons": 0.20, "Stability over seeds": 0.15, "Actionable counterfactuals": 0.10,
           "Runtime for nightly batch": 0.05, "Forms covered": 0.10, "Works with the approved model": 0.10,
           "License for production": 0.10, "Maintenance": 0.10, "Reproducibility": 0.10}
w = np.array([WEIGHTS[k] for k in scores.index]); assert abs(w.sum() - 1) < 1e-9
totals = pd.Series(w @ scores.values, index=scores.columns)
totals.round(2)

A: SHAP + DiCE    4.68
B: EBM + DiCE     4.40
C: LIME + DiCE    3.60
D: HermAI         3.15
dtype: float64

In [6]:
rng = np.random.default_rng(0)
W = rng.dirichlet(w * 40, size=10000)          # 10,000 plausible variations around the chosen weights
first = pd.Series(np.bincount((W @ scores.values).argmax(1), minlength=4) / len(W), index=scores.columns)
print("share of weight variations in which each toolset ranks first:"); print(first.round(3))
print("\ntotals if 'Works with the approved model' had another weight (the others rescaled):")
for x in [0.0, 0.025, 0.05, 0.10, 0.20]:
    ww = w.copy(); i = list(scores.index).index("Works with the approved model")
    ww = ww * (1 - x) / (1 - ww[i]); ww[i] = x
    print(f"  {x:.3f}: totals", {k: round(float(v), 2) for k, v in zip(scores.columns, ww @ scores.values)})

share of weight variations in which each toolset ranks first:
A: SHAP + DiCE    0.937
B: EBM + DiCE     0.063
C: LIME + DiCE    0.000
D: HermAI         0.000
dtype: float64

totals if 'Works with the approved model' had another weight (the others rescaled):
  0.000: totals {'A: SHAP + DiCE': 4.64, 'B: EBM + DiCE': 4.78, 'C: LIME + DiCE': 3.44, 'D: HermAI': 2.94}
  0.025: totals {'A: SHAP + DiCE': 4.65, 'B: EBM + DiCE': 4.68, 'C: LIME + DiCE': 3.48, 'D: HermAI': 2.99}
  0.050: totals {'A: SHAP + DiCE': 4.66, 'B: EBM + DiCE': 4.59, 'C: LIME + DiCE': 3.52, 'D: HermAI': 3.04}
  0.100: totals {'A: SHAP + DiCE': 4.67, 'B: EBM + DiCE': 4.4, 'C: LIME + DiCE': 3.6, 'D: HermAI': 3.15}
  0.200: totals {'A: SHAP + DiCE': 4.71, 'B: EBM + DiCE': 4.02, 'C: LIME + DiCE': 3.75, 'D: HermAI': 3.35}


**Checkpoint 2.** With the team's weights the totals are A 4.68, B 4.40, C 3.60 and D 3.15. A ranks first in about 94 percent of the weight variations and B in about 6 percent. If 'Works with the approved model' had a weight of 2.5 percent or less, B would lead.

## Questions

1. Which toolset would you recommend, and under which written condition? Write it as the first paragraph of a tool memo.
2. Toolset B scores 1 on "Works with the approved model". Is that a property of the tool or of the bank's situation? Would a bank building a new model score it the same way?
3. Choose one score in the table that you think is wrong and rewrite its rule. Does the recommendation change?
4. What would you re-measure, and when, after the memo is signed?